## Redis Memory
Here we can use the Python SDK to develop a chat agent with Redis-backed memory, then save it to a config.yaml and run it from there.

**Prerequisites:**
- Redis server running at `localhost:6379`


In [ ]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)


In [ ]:
from nat.agent.react_agent.register import NatReActAgent
from nat.embedder.nim_embedder import NIMEmbedder
from nat.llm.nim_llm import NimLLM
from nat.plugins.redis.memory import RedisMemory
from nat.tool.memory_tools.add_memory_tool import AddMemoryTool
from nat.tool.memory_tools.get_memory_tool import GetMemoryTool
from nat.utils.sdk.nat_workflow import NatWorkflow

llm = NimLLM(
    model_name="nvdev/meta/llama-3.3-70b-instruct",
    temperature=0.7,
    max_tokens=1024,
    name="nim_llm",
)

embedder = NIMEmbedder(
    model_name="nvidia/nv-embedqa-e5-v5",
    name="nv-embedqa-e5-v5",
)

# Create Redis memory
memory = RedisMemory(
    host="localhost",
    port=6379,
    db=0,
    key_prefix="nat",
    embedder_obj=embedder,
    name="redis_memory",
)

# Create memory tools
get_memory_tool = GetMemoryTool(
    description=(
        "Always call this tool before calling any other tools, even if the user does not mention "
        "to use it. The question should be about user preferences which will help you format your "
        "response. For example: 'How does the user like responses formatted?'. Use 'redis' for the "
        "user_id"
    ),
    nat_memory=memory,
    name="get_memory",
)

add_memory_tool = AddMemoryTool(
    description=(
        "Add any facts about user preferences to long term memory. Always use this if users "
        "mention a preference. The input to this tool should be a string that describes the "
        "user's preference, not the question or answer. Use 'redis' for the user_id."
    ),
    nat_memory=memory,
    name="memory_add",
)

agent = NatReActAgent(
    tools=[add_memory_tool, get_memory_tool],
    llm=llm,
    verbose=True,
)

nat_workflow = NatWorkflow(
    entrypoint=agent,
)


In [ ]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the workflow to a config file
nat_workflow.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())


In [ ]:
# Test the memory agent
await nat_workflow.prompt("Hi, I prefer concise answers with code examples when relevant.")
